<a href="https://colab.research.google.com/github/vanditha18/NLP-Notebooks/blob/main/Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
from nltk.stem.snowball import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer
import nltk
import ast
import spacy
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


import gensim
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

In [ ]:
csv_path = "/content/IMDB_Dataset.csv"

df = pd.read_csv(csv_path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


## Helper functions

In [ ]:
def clean_text(text):
    # Remove HTML tags
    text = re.compile('<.*?>').sub('', text)
    # Remove URL's
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+', '', text)
    # Remove punctuation and digits
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return text

In [ ]:
def get_corpus_and_vocab(documents):
    corpus = []
    for document in documents:
        for word in document.split():
            corpus.append(word)
    return len(corpus), len(set(corpus))

## Text pre-processing

In [ ]:
df['review'] = df['review'].str.lower()
df['review'] = df['review'].apply(clean_text)
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


In [ ]:
# Remove stopwords
nltk.download('stopwords')
stopwords = stopwords.words('english')
stopwords
df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stopwords)]))
df

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,sentiment
0,one reviewers mentioned watching oz episode yo...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive
...,...,...
49995,thought movie right good job wasnt creative or...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,catholic taught parochial elementary schools n...,negative
49998,im going disagree previous comment side maltin...,negative


In [ ]:
df['review'] = df['review'].dropna()
df['review']

,review
0,one reviewers mentioned watching oz episode yo...
1,wonderful little production filming technique ...
2,thought wonderful way spend time hot summer we...
3,basically theres family little boy jake thinks...
4,petter matteis love time money visually stunni...
...,...
49995,thought movie right good job wasnt creative or...
49996,bad plot bad dialogue bad acting idiotic direc...
49997,catholic taught parochial elementary schools n...
49998,im going disagree previous comment side maltin...


## Text Classification

In [ ]:
df

,review,sentiment
0,one reviewers mentioned watching oz episode yo...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive
...,...,...
49995,thought movie right good job wasnt creative or...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,catholic taught parochial elementary schools n...,negative
49998,im going disagree previous comment side maltin...,negative


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.duplicated().sum()

np.int64(422)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df

,review,sentiment
0,one reviewers mentioned watching oz episode yo...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive
...,...,...
49995,thought movie right good job wasnt creative or...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,catholic taught parochial elementary schools n...,negative
49998,im going disagree previous comment side maltin...,negative


In [ ]:
df.duplicated().sum()

np.int64(0)

Preprocessing is done before

In [ ]:
df = df.iloc[:1000]
X = df.iloc[:,0:1]
Y = df['sentiment']

In [ ]:
## Label encoding
le = LabelEncoder()
Y_encoded = le.fit_transform(Y)

In [ ]:
###Split data
X_train, X_test, y_train, y_test = train_test_split(X, Y_encoded, test_size=0.2, random_state=42)

In [ ]:
X_train

,review
29,war movie hollywood genre done redone many tim...
535,film really vile plays urban paranoia ss puts ...
695,okay stupidthey say making another nightmare f...
557,crossfire remains one best hollywood message m...
836,film well advertised avid movie goer seen prev...
...,...
106,performance every actor actress film excellent...
270,clifton webb one favorites however mister scou...
860,production quite surprise absolutely love obsc...
435,wear best italian suits armani hand stitched f...


In [ ]:
## Bag of words technique
cv = CountVectorizer()
X_train_bow = cv.fit_transform(X_train['review'])
X_test_bow = cv.transform(X_test['review'])

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train_bow.toarray(), y_train)

GaussianNB()

In [ ]:
y_pred = gnb.predict(X_test_bow.toarray())

In [ ]:
y_pred

array([0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0,
       1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
       0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0,
       0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0,
       0, 1])

In [ ]:
accuracy_score(y_test, y_pred)

0.575

In [ ]:
confusion_matrix(y_test, y_pred)

array([[57, 47],
       [38, 58]])

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_bow.toarray(), y_train)

RandomForestClassifier()

In [ ]:
y_pred = rf.predict(X_test_bow.toarray())

In [ ]:
accuracy_score(y_test, y_pred)


0.775

In [ ]:
confusion_matrix(y_test, y_pred)

array([[83, 21],
       [24, 72]])

In [ ]:
features = cv.get_feature_names_out()
print(len(features))   # should be <= max_features

18357


In [ ]:
## Bag of words technique with max_features
cv = CountVectorizer(max_features=3000)
X_train_bow = cv.fit_transform(X_train['review'])
X_test_bow = cv.transform(X_test['review'])

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train_bow.toarray(), y_train)
y_pred = gnb.predict(X_test_bow.toarray())
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.61
[[80 24]
 [54 42]]


In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_bow.toarray(), y_train)
y_pred = rf.predict(X_test_bow.toarray())
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.825
[[90 14]
 [21 75]]


In [ ]:
### N-gram
cv = CountVectorizer(ngram_range=(1,2),max_features=5000)
X_train_bow = cv.fit_transform(X_train['review'])
X_test_bow = cv.transform(X_test['review'])

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train_bow.toarray(), y_train)
y_pred = gnb.predict(X_test_bow.toarray())
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.62
[[76 28]
 [48 48]]


In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_bow.toarray(), y_train)
y_pred = rf.predict(X_test_bow.toarray())
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.8
[[86 18]
 [22 74]]


In [ ]:
### TF-IDF

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train['review'])
X_test_tfidf = tfidf.transform(X_test['review'])

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_tfidf.toarray(), y_train)
y_pred = rf.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.81
[[86 18]
 [20 76]]


In [ ]:
### Word2Vec
reviews = []

for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        reviews.append(simple_preprocess(sent))

In [ ]:
model = gensim.models.Word2Vec(window=10, min_count=2, workers=4)
model.build_vocab(reviews)
model.train(reviews, total_examples=model.corpus_count, epochs=model.epochs)

(511913, 591940)

In [ ]:
len(model.wv.index_to_key)

9308

In [ ]:
def document_vector(doc):
  ## Handle OOV words
  doc = [word for word in doc.split() if word in model.wv.index_to_key]
  return np.mean(model.wv[doc],axis=0)

In [ ]:
X = []
for doc in tqdm(df['review'].values):
  X.append(document_vector(doc))

100%|██████████| 1000/1000 [00:07<00:00, 141.39it/s]


In [ ]:
###Split data
X_train, X_test, y_train, y_test = train_test_split(X, Y_encoded, test_size=0.2, random_state=42)

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.555
[[75 29]
 [60 36]]


In [ ]:
import joblib

joblib.dump(gnb, 'gnb_model.pkl')

## Load model
model = joblib.load('gnb_model.pkl')
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.555
[[75 29]
 [60 36]]
